In [ ]:
import fitz
import numpy as np
import qrcode
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import einops

In [ ]:
import pandas as pd
import math

In [ ]:
def get_qrcode(text):
    qr = qrcode.QRCode(border=0)  # Small border for fitting
    qr.add_data(text)
    qr.make(fit=True)
    return qr.make_image(fill="black", back_color="white").convert("RGB")

In [ ]:
import io
import math
from tqdm import tqdm


def _pil_to_bytes(img, fmt="PNG", jpeg_quality=85):
    if fmt.upper() in ("JPG", "JPEG") and img.mode not in ("RGB", "L"):
        img = img.convert("RGB")
    buf = io.BytesIO()
    kw = {}
    if fmt.upper() in ("JPG", "JPEG"):
        kw.update(dict(quality=int(jpeg_quality), optimize=True))
    img.save(buf, format=fmt, **kw)
    return buf.getvalue()


def shrink_rect_to_square(rect: fitz.Rect, pad: float = 0.0, scale: float = 1.0) -> fitz.Rect:
    """
    Return a centered SQUARE rect inside `rect`.

    Steps:
    1) Apply padding (pt) on all sides
    2) Take the largest possible square (side = min(width, height))
    3) Optionally shrink around center by `scale` (<1)

    Guaranteed:
    - Returned rect is square
    - Center-aligned with input rect
    """

    # 1) padded rect
    r = fitz.Rect(rect)
    if pad:
        r.x0 += pad
        r.y0 += pad
        r.x1 -= pad
        r.y1 -= pad

    # guard
    if r.width <= 0 or r.height <= 0:
        return fitz.Rect(0, 0, 0, 0)

    # 2) largest centered square
    side = min(r.width, r.height)

    # 3) optional scale
    side *= scale

    cx = (r.x0 + r.x1) * 0.5
    cy = (r.y0 + r.y1) * 0.5
    half = side * 0.5

    return fitz.Rect(cx - half, cy - half, cx + half, cy + half)

def tile_pil_df_to_pdf(
    df, img_col, text1_col, text2_col, out_pdf,
    nr=4, nc=4,
    page_size="letter", landscape=False,
    margin=36, gutter=8,
    keep_aspect=True,
    text_fontsize=8, text_leading=1.15, text_pad=2, text_align="center",
    img_format="PNG", jpeg_quality=85,
    img_pad=1,          # NEW: inner padding inside image box (pt)
    img_scale=0.5,     # NEW: additional shrink factor (<1) inside image box
    tile_border_width=0.8,          # pt
    tile_border_color=(0, 0, 0),    # RGB in [0,1]
    tile_border_overlay=True,       # draw after content
):
    
    FONT_NAME = "RobotoMono"
    FONT_FILE = "assets/RobotoMono-Regular.ttf"


    imgs = df[img_col].tolist()
    texts1 = df[text1_col].fillna("").astype(str).tolist()
    texts2 = df[text2_col].fillna("").astype(str).tolist()
    img_bytes = [_pil_to_bytes(im, fmt=img_format, jpeg_quality=jpeg_quality) for im in imgs]

    NI = len(img_bytes)
    doc = fitz.open()

    if isinstance(page_size, str):
        w_pt, h_pt = fitz.paper_size(page_size)
    else:
        w_pt, h_pt = page_size
    if landscape:
        w_pt, h_pt = max(w_pt, h_pt), min(w_pt, h_pt)

    tiles_per_page = nr * nc
    n_pages = math.ceil(NI / tiles_per_page)

    content_w = w_pt - 2 * margin
    content_h = h_pt - 2 * margin
    tile_w = (content_w - (nc - 1) * gutter) / nc
    tile_h = (content_h - (nr - 1) * gutter) / nr

    line_h = text_fontsize * text_leading
    text_band_h = 2 * line_h + 2 * text_pad
    if tile_h <= text_band_h:
        raise ValueError("Text band too tall for tile height. Reduce fontsize/nr or increase page size.")

    align_map = {"left": fitz.TEXT_ALIGN_LEFT, "center": fitz.TEXT_ALIGN_CENTER, "right": fitz.TEXT_ALIGN_RIGHT}
    align = align_map.get(text_align, fitz.TEXT_ALIGN_CENTER)

    tile_rects = []
    for r in range(nr):
        for c in range(nc):
            x0 = margin + c * (tile_w + gutter)
            y0 = margin + r * (tile_h + gutter)
            tile_rects.append(fitz.Rect(x0, y0, x0 + tile_w, y0 + tile_h))

    idx = 0
    for _ in range(n_pages):
        page = doc.new_page(width=w_pt, height=h_pt)
        
        # once per page (right after new_page)
        page.insert_font(fontname=FONT_NAME, fontfile=FONT_FILE)


    
        for k in range(tiles_per_page):
            if idx >= NI:
                break

            tile = tile_rects[k]
            page.draw_rect(tile, width=tile_border_width, color=tile_border_color, fill=None)
            img_rect_base = fitz.Rect(tile.x0, tile.y0, tile.x1, tile.y1 - text_band_h)
            img_rect = shrink_rect_to_square(img_rect_base, pad=img_pad, scale=img_scale)

            band_top = tile.y1 - text_band_h
            t1_rect = fitz.Rect(tile.x0 + text_pad, band_top + text_pad-2,
                                tile.x1 - text_pad, band_top + text_pad + line_h-2)
            
            t2_rect = fitz.Rect(tile.x0 + text_pad, band_top + text_pad + line_h-10,
                                tile.x1 - text_pad, tile.y1 - text_pad-10)

            page.insert_image(img_rect, stream=img_bytes[idx], keep_proportion=keep_aspect, overlay=True)

            rc1 = page.insert_textbox(t1_rect, texts1[idx], fontsize=text_fontsize,
                                      fontname=FONT_NAME,
                                      align=align)
            rc2 = page.insert_textbox(t2_rect, texts2[idx], fontsize=text_fontsize,
                                      fontname=FONT_NAME,
                                      align=align)

            # If negative => nothing rendered (didn't fit)
            if t1_rect.width <= 0 or t1_rect.height <= 0:
                print("BAD RECT", t1_rect, "tile", tile)
            if rc1 < 0 or rc2 < 0:
                print("NO FIT", idx, rc1, rc2, t1_rect, t2_rect, texts1[idx][:40], texts2[idx][:40])
            idx += 1

    doc.save(out_pdf, deflate=True)
    doc.close()
    return out_pdf

# Main

In [ ]:
to_pull = pd.read_csv("data/topull_phase3.csv")

In [ ]:
to_pull["anon_id"] = np.arange(len(to_pull))
to_pull["anon_id"] = to_pull["anon_id"].apply(lambda i: f"MLiNS-C-{i:04d}")

In [ ]:
to_pull["qrcode"] = to_pull["anon_id"].apply(get_qrcode)

In [ ]:
to_pull[["su", "anon_id"]]#.to_csv("data/phase3_mapping.csv", index=False)

In [ ]:
tile_pil_df_to_pdf(
    to_pull,
    img_col="qrcode",
    text1_col="su",
    text2_col="anon_id",
    out_pdf="test_out_0209.pdf",
    nr=6,
    nc=4,
    page_size="letter",     # or (w_pt, h_pt)
    landscape=False,
    margin=36,
    gutter=8,
    keep_aspect=True,
    text_fontsize=10,
    text_leading=2,
    text_pad=0.5,
    text_align="center",    # "left"|"center"|"right"
    img_pad=0,          # NEW: inner padding inside image box (pt)
    img_scale=0.8,
)